In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/test_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/val_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/train_80_mode_no_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/test_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/val_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/train_80_mode_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/train_80_missing_no_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/val_10.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/test_10.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/train_80_missing_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing t

In [2]:
# ============================================================
# CARDIAC MI STAGING — Bio+Discharge Summary BERT (Text-ified Tabular)
# TARGET: Phase (Non-MI, Chronic, Sub-acute, Acute)
# ============================================================
import subprocess, sys, os, warnings
warnings.filterwarnings("ignore")

def pip_install(pkg):
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"[OK] {pkg}")
    else:
        print(f"[WARN] {pkg}: {result.stderr[-200:]}")

pip_install("transformers")
pip_install("datasets")
pip_install("accelerate")
pip_install("evaluate")

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset

# ====================== CONFIG ======================
TRAIN_PATH = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/train_80_missing_smote.xlsx"
TEST_PATH  = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/test_10.xlsx"
TARGET_COL = "Phase"
RANDOM_SEED = 42

# ── Top-10 features from feature importance chart ──────────────────────────
SELECTED_FEATURES = [
    "Blood_Glucose_mmolL",
    "Diastolic_BP",
    "Chest_Pain",
    "Weighted_Risk_Score",
    "Systolic_BP",
    "Heart_Rate_bpm",
    "Weighted_Symptom_Score",
    "Serum_Creatinine_mgdL",
    "hs_Troponin_I_ngL",
    "Age_Adjusted_Troponin",
]
# ───────────────────────────────────────────────────────────────────────────

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Device: {DEVICE}")
print(f"Selected features ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}")

# ============================================================
# DATA LOADING & TEXT SERIALIZATION
# ============================================================
def load_and_serialize(train_path, test_path):
    train_df = pd.read_excel(train_path)
    test_df  = pd.read_excel(test_path)

    # ── Auto-rename target if missing in test set ──────────────────────────
    if TARGET_COL not in test_df.columns:
        for col in test_df.columns:
            if test_df[col].astype(str).isin(
                train_df[TARGET_COL].astype(str).unique()
            ).sum() > 5:
                test_df = test_df.rename(columns={col: TARGET_COL})
                print(f"[INFO] Auto-renamed target to '{TARGET_COL}'")
                break

    print(f"Train shape (raw): {train_df.shape} | Test shape (raw): {test_df.shape}")

    # ── Validate that all selected features exist ──────────────────────────
    missing_train = [f for f in SELECTED_FEATURES if f not in train_df.columns]
    missing_test  = [f for f in SELECTED_FEATURES if f not in test_df.columns]
    if missing_train:
        raise ValueError(f"Missing in train: {missing_train}")
    if missing_test:
        raise ValueError(f"Missing in test:  {missing_test}")

    # ── Keep only selected features + target ──────────────────────────────
    train_df = train_df[SELECTED_FEATURES + [TARGET_COL]]
    test_df  = test_df[SELECTED_FEATURES  + [TARGET_COL]]

    print(f"Train shape (filtered): {train_df.shape} | Test shape (filtered): {test_df.shape}")
    print(f"Train Phase distribution:\n{train_df[TARGET_COL].value_counts()}\n")

    # ── Encode labels ──────────────────────────────────────────────────────
    le = LabelEncoder()
    le.fit(train_df[TARGET_COL])
    y_train = le.transform(train_df[TARGET_COL])
    y_test  = le.transform(test_df[TARGET_COL])

    global CLASS_NAMES
    CLASS_NAMES = le.classes_.tolist()
    print(f"Classes: {CLASS_NAMES}")

    # ── Features only ─────────────────────────────────────────────────────
    X_train = train_df[SELECTED_FEATURES]
    X_test  = test_df[SELECTED_FEATURES]

    # ── Convert tabular row → natural text ────────────────────────────────
    def row_to_text(row):
        return " | ".join(
            [f"{col}:{val}" for col, val in row.items() if pd.notna(val)]
        )

    train_texts = X_train.apply(row_to_text, axis=1).tolist()
    test_texts  = X_test.apply(row_to_text, axis=1).tolist()

    # Preview
    print(f"\nSample serialized text:\n  {train_texts[0]}\n")

    return train_texts, test_texts, y_train, y_test, CLASS_NAMES


train_texts, test_texts, y_train, y_test, CLASS_NAMES = load_and_serialize(
    TRAIN_PATH, TEST_PATH
)

# ============================================================
# Bio+Discharge Summary BERT SETUP
# ============================================================
print("\n" + "█"*70)
print(" MODEL: Bio+Discharge Summary BERT")
print("█"*70)

MODEL_NAME = "emilyalsentzer/Bio_Discharge_Summary_BERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Hugging Face Datasets ─────────────────────────────────────────────────
train_data = Dataset.from_dict({"text": train_texts, "label": y_train.tolist()})
test_data  = Dataset.from_dict({"text": test_texts,  "label": y_test.tolist()})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256,   # 10 features → short texts; 256 is sufficient
    )

train_data = train_data.map(tokenize_function, batched=True)
test_data  = test_data.map(tokenize_function, batched=True)

train_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format("torch",  columns=["input_ids", "attention_mask", "label"])

# ── Model ─────────────────────────────────────────────────────────────────
num_labels = len(CLASS_NAMES)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
).to(DEVICE)

# ====================== TRAINING ARGS ======================
training_args = TrainingArguments(
    output_dir="./discharge_summary_bert_results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    seed=RANDOM_SEED,
    remove_unused_columns=False,
)

# ====================== METRICS ======================
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall":    recall_score(labels, preds, average="weighted", zero_division=0),
    }

# ====================== TRAINER ======================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

# ====================== TRAINING ======================
print("Starting Bio+Discharge Summary BERT fine-tuning on selected features...")
trainer.train()

# ====================== EVALUATION ======================
print("\nEvaluating on test set...")
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

# ── Final Predictions ─────────────────────────────────────────────────────
predictions = trainer.predict(test_data)
preds = np.argmax(predictions.predictions, axis=1)

print("\n" + "="*70)
print("Bio+Discharge Summary BERT FINAL RESULTS  (top-10 features only)")
print("="*70)
print(classification_report(y_test, preds, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_test, preds)
print("\nConfusion Matrix:\n", cm)

print("\n✅ Bio+Discharge Summary BERT training & evaluation completed!")

[OK] transformers
[OK] datasets
[OK] accelerate
[OK] evaluate
Device: cuda
Selected features (10): ['Blood_Glucose_mmolL', 'Diastolic_BP', 'Chest_Pain', 'Weighted_Risk_Score', 'Systolic_BP', 'Heart_Rate_bpm', 'Weighted_Symptom_Score', 'Serum_Creatinine_mgdL', 'hs_Troponin_I_ngL', 'Age_Adjusted_Troponin']
Train shape (raw): (1776, 23) | Test shape (raw): (190, 23)
Train shape (filtered): (1776, 11) | Test shape (filtered): (190, 11)
Train Phase distribution:
Phase
Acute        520
Non-MI       434
Chronic      422
Sub-acute    400
Name: count, dtype: int64

Classes: ['Acute', 'Chronic', 'Non-MI', 'Sub-acute']

Sample serialized text:
  Blood_Glucose_mmolL:12.7 | Diastolic_BP:-0.2367988898819029 | Chest_Pain:1.0 | Weighted_Risk_Score:7.0 | Systolic_BP:-0.3695581856124019 | Heart_Rate_bpm:1.223287249716773 | Weighted_Symptom_Score:7.0 | Serum_Creatinine_mgdL:1.15 | hs_Troponin_I_ngL:1.774542798744036 | Age_Adjusted_Troponin:2.113646961457278


█████████████████████████████████████████████

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1776 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on test set...


Evaluation Results: {'eval_loss': 0.3027346432209015, 'eval_accuracy': 0.9578947368421052, 'eval_f1': 0.9578947368421052, 'eval_precision': 0.9578947368421052, 'eval_recall': 0.9578947368421052, 'eval_runtime': 1.9106, 'eval_samples_per_second': 99.444, 'eval_steps_per_second': 3.14, 'epoch': 5.0}

Bio+Discharge Summary BERT FINAL RESULTS  (top-10 features only)
              precision    recall  f1-score   support

       Acute       0.94      0.94      0.94        65
     Chronic       1.00      1.00      1.00        52
      Non-MI       1.00      1.00      1.00        55
   Sub-acute       0.78      0.78      0.78        18

    accuracy                           0.96       190
   macro avg       0.93      0.93      0.93       190
weighted avg       0.96      0.96      0.96       190


Confusion Matrix:
 [[61  0  0  4]
 [ 0 52  0  0]
 [ 0  0 55  0]
 [ 4  0  0 14]]

✅ Bio+Discharge Summary BERT training & evaluation completed!
